# Step 1: Understanding the database

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv("../data/dataset.csv")
print(df.head())

   Order_ID  Distance_km Weather Traffic_Level Time_of_Day Vehicle_Type  \
0       522         7.93   Windy           Low   Afternoon      Scooter   
1       738        16.42   Clear        Medium     Evening         Bike   
2       741         9.52   Foggy           Low       Night      Scooter   
3       661         7.44   Rainy        Medium   Afternoon      Scooter   
4       412        19.03   Clear           Low     Morning         Bike   

   Preparation_Time_min  Courier_Experience_yrs  Delivery_Time_min  
0                    12                     1.0                 43  
1                    20                     2.0                 84  
2                    28                     1.0                 59  
3                     5                     1.0                 37  
4                    16                     5.0                 68  


In [4]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 1000
Columns: 9


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Order_ID                1000 non-null   int64  
 1   Distance_km             1000 non-null   float64
 2   Weather                 970 non-null    str    
 3   Traffic_Level           970 non-null    str    
 4   Time_of_Day             970 non-null    str    
 5   Vehicle_Type            1000 non-null   str    
 6   Preparation_Time_min    1000 non-null   int64  
 7   Courier_Experience_yrs  970 non-null    float64
 8   Delivery_Time_min       1000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 91.3 KB


In [10]:
missing_data = pd.DataFrame({
    "Missing_values": df.isnull().sum(),
    "Percentage": df.isnull().mean() * 100
})

missing_data

,Missing_values,Percentage
Order_ID,0,0.0
Distance_km,0,0.0
Weather,30,3.0
Traffic_Level,30,3.0
Time_of_Day,30,3.0
Vehicle_Type,0,0.0
Preparation_Time_min,0,0.0
Courier_Experience_yrs,30,3.0
Delivery_Time_min,0,0.0


In [ ]:
categorical_columns = [
    "Weather",
    "Traffic_Level",
    "Time_of_Day",
    "Vehicle_Type"
]

for column in categorical_columns:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))

Weather
Clear    470
Rainy    204
Foggy    103
Snowy     97
Windy     96
NaN       30
Name: count, dtype: int64
Traffic_Level
Medium    390
Low       383
High      197
NaN        30
Name: count, dtype: int64
Time_of_Day
Morning      308
Evening      293
Afternoon    284
Night         85
NaN           30
Name: count, dtype: int64
Vehicle_Type
Bike       503
Scooter    302
Car        195
Name: count, dtype: int64


In [11]:
numerical_columns = [
    "Distance_km",
    "Preparation_Time_min",
    "Courier_Experience_yrs",
    "Delivery_Time_min"
]

df[numerical_columns].describe()

,Distance_km,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
count,1000.000000,1000.000000,970.000000,1000.000000
mean,10.059970,16.982000,4.579381,56.732000
std,5.696656,7.204553,2.914394,22.070915
min,0.590000,5.000000,0.000000,8.000000
25%,5.105000,11.000000,2.000000,41.000000
50%,10.190000,17.000000,5.000000,55.500000
75%,15.017500,23.000000,7.000000,71.000000
max,19.990000,29.000000,9.000000,153.000000


In [12]:
print("Duplicated rows:", df.duplicated().sum())

print(
    "Duplicated Order_ID:",
    df["Order_ID"].duplicated().sum()
)

Duplicated rows: 0
Duplicated Order_ID: 0


In [13]:
print(
    "Invalid distance:",
    (df["Distance_km"] <= 0).sum()
)

print(
    "Invalid preparation time:",
    (df["Preparation_Time_min"] <= 0).sum()
)

print(
    "Invalid courier experience:",
    (df["Courier_Experience_yrs"] < 0).sum()
)

print(
    "Invalid delivery time:",
    (df["Delivery_Time_min"] <= 0).sum()
)

Invalid distance: 0
Invalid preparation time: 0
Invalid courier experience: 0
Invalid delivery time: 0


In [15]:
def detect_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    print("Column:", column)
    print("Lower bound:", lower_bound)
    print("Upper bound:", upper_bound)
    print("Number of outliers:", len(outliers))

    return outliers

detect_outliers(df, "Distance_km")
detect_outliers(df, "Preparation_Time_min")
detect_outliers(df, "Courier_Experience_yrs")
delivery_outliers = detect_outliers(
    df,
    "Delivery_Time_min"
)
delivery_outliers

Column: Distance_km
Lower bound: -9.763749999999998
Upper bound: 29.886249999999997
Number of outliers: 0
Column: Preparation_Time_min
Lower bound: -7.0
Upper bound: 41.0
Number of outliers: 0
Column: Courier_Experience_yrs
Lower bound: -5.5
Upper bound: 14.5
Number of outliers: 0
Column: Delivery_Time_min
Lower bound: -4.0
Upper bound: 116.0
Number of outliers: 6


,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
29,948,18.05,Clear,Medium,Evening,Scooter,10,7.0,123
127,446,18.97,Clear,Low,Evening,Car,25,4.0,141
379,814,18.46,Clear,NaN,NaN,Scooter,29,1.0,153
452,394,15.64,Rainy,Low,NaN,Bike,20,4.0,141
784,385,14.83,Rainy,Low,Morning,Car,19,4.0,126
924,428,17.81,Windy,High,Evening,Bike,21,4.0,122


## Cleaning decisions

- `Weather`: 30 missing values. Will be imputed using the most frequent category.
- `Traffic_Level`: 30 missing values. Will be imputed using the most frequent category.
- `Time_of_Day`: 30 missing values. Will be imputed using the most frequent category.
- `Courier_Experience_yrs`: 30 missing values. Will be imputed using the median.

- `Distance_km`: no invalid values or significant outliers detected.
- `Preparation_Time_min`: no invalid values or significant outliers detected.
- `Vehicle_Type`: categories are valid and consistent.
- `Delivery_Time_min`: six statistical outliers detected, but retained because they represent plausible delayed deliveries.
- `Order_ID`: retained for traceability but will not be used as a predictive feature.
- No duplicate rows or duplicate order IDs were detected.

Missing-value imputation will be performed later inside a scikit-learn Pipeline to avoid data leakage.

# Step 2 : Exploratory Data Analysis

In [16]:
df["Delivery_Time_min"].describe()

count    1000.000000
mean       56.732000
std        22.070915
min         8.000000
25%        41.000000
50%        55.500000
75%        71.000000
max       153.000000
Name: Delivery_Time_min, dtype: float64